# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanzina-Aranya-Islam/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

The paper reports that pages refreshed within 30 days in the 365+ day group showed a 3.2x health improvement and 57x more impressions than the comparison group.

**Methodology question:** Where does the outcome come from, and does the comparison support the strength of the claim? The outcome is based on observed health and impression differences between freshness groups in the portfolio. I would want to confirm how the comparison groups were constructed and whether pages were comparable before the refresh. Because this is observational data, the measured difference supports an association in this portfolio, but it does not by itself establish that refreshing a page caused the improvement. A stronger causal claim would need a clearly defined pre/post or matched comparison design with a future outcome window.

### Finding 2 — Random Forest: What Predicts Health?

The ML appendix reports a Random Forest model for health score, with Average Position and Impressions receiving the highest feature importance.

**Methodology question:** Where does the label come from, and does the validation design support the claim? The target is the FlyRank health score, which is partly constructed from inputs including impressions and average position. The paper states that the model was holdout-tested with an 80/20 split, but I would want to know whether related observations from the same client could appear in both sets. More importantly, because some model features are components of the health-score target itself, high feature importance is expected and should not be interpreted as evidence that changing those features will cause health to improve. The paper appropriately describes this result as exploratory and descriptive rather than causal.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Paper claim audit completed.")
print("Finding 1: Freshness Multiplier — observational comparison; causal interpretation should be limited.")
print("Finding 2: Random Forest health model — target partly constructed from model inputs; importance is descriptive.")

Paper claim audit completed.
Finding 1: Freshness Multiplier — observational comparison; causal interpretation should be limited.
Finding 2: Random Forest health model — target partly constructed from model inputs; importance is descriptive.


## 2. My model under an honest split (before/after)

I compared two validation designs using the same Random Forest model, features, target, and evaluation metrics.

**Before:** a row-level random 80/20 split. This can place records from the same client in both training and test sets, which may make performance look more optimistic for unseen-client generalization.

**After:** an 80/20 client-grouped split using `client_hash_id`. This keeps each client entirely in either the training or test set.

The main change is therefore the validation design, not the model itself. I interpret the difference as evidence about validation sensitivity rather than as proof of model quality or causation.

### Before/after result

The row-level random split measured ROC-AUC of 0.9690 and PR-AUC of 0.6291.

The client-grouped split measured ROC-AUC of 0.9732 and PR-AUC of 0.7164, with zero client overlap between training and test sets.

In this audit, the grouped split did not reduce measured performance. Therefore, I do not claim that the original validation was inflated. The grouped split is still preferred because it provides a cleaner test of generalization to unseen clients.

In [10]:
import pandas as pd

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Rows:", len(df_march))
print("Columns:", df_march.columns.tolist())

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

audit_df = df_march.copy()

features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai"
]
target = "clicked"

audit_df[target] = (audit_df["gsc_clicks"] > 0).astype(int)


audit_df = audit_df[
    features + [target, "client_hash_id"]
].copy()

audit_df[features] = audit_df[features].fillna(0)


sample_n = min(500_000, len(audit_df))
audit_df = audit_df.sample(n=sample_n, random_state=42)

X = audit_df[features]
y = audit_df[target]
groups = audit_df["client_hash_id"]


def make_model():
    return RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )


X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_before = make_model()
model_before.fit(X_train_b, y_train_b)

pred_before = model_before.predict_proba(X_test_b)[:, 1]

roc_before = roc_auc_score(y_test_b, pred_before)
pr_before = average_precision_score(y_test_b, pred_before)

print("BEFORE — Row-level random split")
print(f"Train rows: {len(X_train_b):,}")
print(f"Test rows:  {len(X_test_b):,}")
print(f"ROC-AUC:    {roc_before:.4f}")
print(f"PR-AUC:     {pr_before:.4f}")



gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_a = X.iloc[train_idx]
X_test_a = X.iloc[test_idx]
y_train_a = y.iloc[train_idx]
y_test_a = y.iloc[test_idx]

train_clients = groups.iloc[train_idx]
test_clients = groups.iloc[test_idx]

model_after = make_model()
model_after.fit(X_train_a, y_train_a)

pred_after = model_after.predict_proba(X_test_a)[:, 1]

roc_after = roc_auc_score(y_test_a, pred_after)
pr_after = average_precision_score(y_test_a, pred_after)

overlap = set(train_clients).intersection(set(test_clients))

print("\nAFTER — Client-grouped split")
print(f"Train rows: {len(X_train_a):,}")
print(f"Test rows:  {len(X_test_a):,}")
print(f"Train clients: {train_clients.nunique():,}")
print(f"Test clients:  {test_clients.nunique():,}")
print(f"Client overlap: {len(overlap)}")
print(f"ROC-AUC:    {roc_after:.4f}")
print(f"PR-AUC:     {pr_after:.4f}")



comparison = pd.DataFrame({
    "Validation": ["Before: random row split", "After: client-grouped split"],
    "ROC_AUC": [roc_before, roc_after],
    "PR_AUC": [pr_before, pr_after]
})

print("\nBefore / After comparison:")
display(comparison.round(4))

print("\nAudit interpretation:")
if roc_after < roc_before:
    print("The grouped split produced lower ROC-AUC, indicating that row-level validation was more optimistic in this audit.")
else:
    print("The grouped split did not produce a lower ROC-AUC in this audit; the validation design still provides a stricter unseen-client test.")

if pr_after < pr_before:
    print("PR-AUC also decreased under client grouping, showing sensitivity to the validation design.")
else:
    print("PR-AUC did not decrease under client grouping in this audit.")

print("Client overlap after grouping:", len(overlap))

BEFORE — Row-level random split
Train rows: 400,000
Test rows:  100,000
ROC-AUC:    0.9690
PR-AUC:     0.6291

AFTER — Client-grouped split
Train rows: 454,219
Test rows:  45,781
Train clients: 44
Test clients:  11
Client overlap: 0
ROC-AUC:    0.9732
PR-AUC:     0.7164

Before / After comparison:


,Validation,ROC_AUC,PR_AUC
0,Before: random row split,0.9690,0.6291
1,After: client-grouped split,0.9732,0.7164



Audit interpretation:
The grouped split did not produce a lower ROC-AUC in this audit; the validation design still provides a stricter unseen-client test.
PR-AUC did not decrease under client grouping in this audit.
Client overlap after grouping: 0


## 3. Leakage audit

I audited the final model features for target leakage, future information, and decision-output leakage.

The final features are:

- `gsc_impressions`
- `gsc_avg_position`
- `ga4_sessions`
- `sessions_ai`

The target is whether `gsc_clicks > 0`.

None of the final features is the target itself, a future outcome, or a Week-4 decision output.

However, these are same-row observed signals from the March 2026 dataset. Therefore, the model should not be described as a future forecasting model. The main modeling limitation is that missing values were filled with zero, which can sometimes mix true zero activity with unavailable data.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit

features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai"
]

target = "gsc_clicks"

print("FINAL FEATURES")
for f in features:
    print(" -", f)

print("\nTARGET")
print(" -", target)

print("\nDirect target leakage:")
print("Target included in features:", target in features)

future_keywords = [
    "future",
    "next",
    "forecast",
    "tomorrow",
    "7d",
    "30d",
    "90d"
]

decision_keywords = [
    "action",
    "priority",
    "recommendation",
    "review",
    "score"
]

print("\nPossible future-information names:")
for f in features:
    matches = [k for k in future_keywords if k in f.lower()]
    print(f, "->", matches if matches else "none")

print("\nPossible decision-output names:")
for f in features:
    matches = [k for k in decision_keywords if k in f.lower()]
    print(f, "->", matches if matches else "none")

print("\nLeakage audit conclusion:")
print("No direct target, future-outcome, or decision-output feature was found.")
print("Important limitation: this is same-row classification, not future forecasting.")

FINAL FEATURES
 - gsc_impressions
 - gsc_avg_position
 - ga4_sessions
 - sessions_ai

TARGET
 - gsc_clicks

Direct target leakage:
Target included in features: False

Possible future-information names:
gsc_impressions -> none
gsc_avg_position -> none
ga4_sessions -> none
sessions_ai -> none

Possible decision-output names:
gsc_impressions -> none
gsc_avg_position -> none
ga4_sessions -> none
sessions_ai -> none

Leakage audit conclusion:
No direct target, future-outcome, or decision-output feature was found.
Important limitation: this is same-row classification, not future forecasting.


## 4. Claim rewrite

The model makes both false-positive and false-negative classifications at the 0.5 threshold. These examples show that the model does not perfectly classify every daily content record.

A false positive is a record predicted as clicked when the observed click target was zero. A false negative is a record with observed clicks that the model predicted as zero.

These errors reinforce that the model is a decision-support tool rather than a definitive explanation of page performance.

**Original-style claim:**
The Random Forest predicts which pages will get clicks with high accuracy.

**Safer claim:**
On the March 2026 dataset, the Random Forest measured ROC-AUC of 0.9732 and PR-AUC of 0.7164 on a client-grouped held-out test set. The model classified whether a daily record had at least one observed GSC click. The result is directional and can support decision-making, but it does not establish future click prediction or causation.

The failure examples also show that the model can be highly confident and still be wrong. This reinforces that the model should be used as decision-support rather than as a definitive explanation of page performance.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Real failure examples from the client-grouped test set
# Real failure examples from the client-grouped test set

test_results = X_test_a.copy()

test_results["actual_clicked"] = y_test_a.to_numpy()
test_results["predicted_probability"] = pred_after
test_results["predicted_clicked"] = (
    test_results["predicted_probability"] >= 0.5
).astype(int)

false_positives = test_results[
    (test_results["actual_clicked"] == 0) &
    (test_results["predicted_clicked"] == 1)
].sort_values(
    "predicted_probability",
    ascending=False
).head(10)

false_negatives = test_results[
    (test_results["actual_clicked"] == 1) &
    (test_results["predicted_clicked"] == 0)
].sort_values(
    "predicted_probability",
    ascending=True
).head(10)

display_columns = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai",
    "actual_clicked",
    "predicted_probability",
    "predicted_clicked"
]

print("FALSE POSITIVE EXAMPLES")
display(false_positives[display_columns])

print("FALSE NEGATIVE EXAMPLES")
display(false_negatives[display_columns])

print("\nFailure-example audit:")
print("False positives shown:", len(false_positives))
print("False negatives shown:", len(false_negatives))
print("Threshold used: 0.5")

FALSE POSITIVE EXAMPLES


,gsc_impressions,gsc_avg_position,ga4_sessions,sessions_ai,actual_clicked,predicted_probability,predicted_clicked
4064000,1895,5.519789,3.0,0.0,0,0.996854,1
5349746,625,3.166400,2.0,0.0,0,0.995196,1
4399440,523,6.380497,2.0,0.0,0,0.994033,1
8230802,268,3.783582,2.0,0.0,0,0.992270,1
84249,213,5.732394,2.0,0.0,0,0.990289,1
2534953,205,5.136585,2.0,0.0,0,0.990158,1
2991678,208,6.793269,3.0,0.0,0,0.989196,1
3741707,224,5.727679,2.0,0.0,0,0.989183,1
7221992,1129,2.388840,1.0,0.0,0,0.989154,1
2199863,232,6.030172,2.0,0.0,0,0.989114,1


FALSE NEGATIVE EXAMPLES


,gsc_impressions,gsc_avg_position,ga4_sessions,sessions_ai,actual_clicked,predicted_probability,predicted_clicked
8822693,1,10.000000,0.0,0.0,1,0.027539,0
4062559,3,51.333333,0.0,0.0,1,0.031439,0
1710253,1,11.000000,0.0,0.0,1,0.035801,0
425445,42,46.214286,0.0,0.0,1,0.053869,0
2200550,16,43.562500,0.0,0.0,1,0.073376,0
1653794,2,6.000000,0.0,0.0,1,0.077486,0
4514178,37,44.378378,0.0,0.0,1,0.085448,0
8131141,85,56.717647,0.0,0.0,1,0.086172,0
8912519,2,8.000000,0.0,0.0,1,0.086815,0
4854473,2,5.000000,0.0,0.0,1,0.090689,0



Failure-example audit:
False positives shown: 10
False negatives shown: 10
Threshold used: 0.5


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.